# Audio HMTL V2 训练 (Colab)

**目标**: 添加 7类情绪输出，与 Text HMTL 统一格式

**输出格式**:
```python
{
    'label_7_logits': [batch_size, 7],   # 7类情绪
    'label_4_logits': [batch_size, 4],   # 4类分类
    'label_3_logits': [batch_size, 3],   # 3类极性
    'arousal': [batch_size],             # 激活度
    'valence': [batch_size]              # 效价
}
```

## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install transformers -q

## 2. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 设置路径 - 请根据你的 Drive 结构修改
DRIVE_PATH = "/content/drive/MyDrive/bigcreate"
CACHE_FILE = f"{DRIVE_PATH}/audio_features_cache.pt"
LABELS_CSV = f"{DRIVE_PATH}/audio_hmtl_labels_v2.csv"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/audio_hmtl_v2_best.pt"

import os
print(f"缓存文件存在: {os.path.exists(CACHE_FILE)}")
print(f"标签文件存在: {os.path.exists(LABELS_CSV)}")

## 3. 复制缓存到本地 (加速训练)

In [ ]:
import shutil

LOCAL_CACHE = "/content/audio_features_cache.pt"

if not os.path.exists(LOCAL_CACHE):
    print("复制缓存到本地...")
    shutil.copy(CACHE_FILE, LOCAL_CACHE)
    print("完成!")
else:
    print("本地缓存已存在")

CACHE_FILE = LOCAL_CACHE

## 4. 定义模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Model, Wav2Vec2Config

WAV2VEC2_MODEL_NAME = "facebook/wav2vec2-base"

class AudioHMTLClassifier(nn.Module):
    """
    Audio HMTL V2 - 统一输出格式
    """
    
    def __init__(self, dropout=0.3):
        super().__init__()
        
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(WAV2VEC2_MODEL_NAME)
        self.wav2vec2.freeze_feature_extractor()
        
        hidden_size = 768
        
        self.dim_reducer = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # 7类情绪分类头
        self.classifier_7 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(128, 7)
        )
        
        # 4类分类头
        self.classifier_4 = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(64, 4)
        )
        
        # 3类极性分类头
        self.classifier_3 = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(64, 3)
        )
        
        # Arousal 回归头
        self.regressor_A = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Valence 回归头
        self.regressor_V = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )
    
    def forward(self, input_values, attention_mask=None):
        output = self.wav2vec2(input_values, attention_mask=attention_mask)
        mean_pooling = torch.mean(output.last_hidden_state, dim=1)
        x = self.dim_reducer(mean_pooling)
        
        return {
            'label_7_logits': self.classifier_7(x),
            'label_4_logits': self.classifier_4(x),
            'label_3_logits': self.classifier_3(x),
            'arousal': self.regressor_A(x).squeeze(-1),
            'valence': self.regressor_V(x).squeeze(-1)
        }

print("模型定义完成!")

## 5. 定义数据集和损失函数

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class CachedAudioDatasetV2(Dataset):
    def __init__(self, label_df, cached_features):
        self.cached_features = cached_features
        self.label_df = label_df
        
        self.valid_indices = []
        for idx in label_df.index:
            if idx in cached_features:
                self.valid_indices.append(idx)
        
        print(f"有效样本: {len(self.valid_indices)}/{len(label_df)}")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        orig_idx = self.valid_indices[idx]
        row = self.label_df.loc[orig_idx]
        
        return {
            'features': self.cached_features[orig_idx],
            'label_7': torch.tensor(row['label_7_emotion'], dtype=torch.long),
            'label_4': torch.tensor(row['label_4_emotion'], dtype=torch.long),
            'label_3': torch.tensor(row['label_3_polarity'], dtype=torch.long),
            'arousal': torch.tensor(row['true_arousal'], dtype=torch.float),
            'valence': torch.tensor(row['true_valence'], dtype=torch.float),
        }


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    
    max_len = 48000  # 3秒
    
    features_list = []
    for item in batch:
        feat = item['features']
        if feat.shape[0] > max_len:
            start = np.random.randint(0, feat.shape[0] - max_len)
            feat = feat[start:start + max_len]
        elif feat.shape[0] < max_len:
            pad_len = max_len - feat.shape[0]
            feat = F.pad(feat, (0, pad_len))
        features_list.append(feat)
    
    return {
        'features': torch.stack(features_list),
        'label_7': torch.stack([b['label_7'] for b in batch]),
        'label_4': torch.stack([b['label_4'] for b in batch]),
        'label_3': torch.stack([b['label_3'] for b in batch]),
        'arousal': torch.stack([b['arousal'] for b in batch]),
        'valence': torch.stack([b['valence'] for b in batch]),
    }


class HMTLAudioLossV2(nn.Module):
    def __init__(self, class_weights_7=None, class_weights_4=None):
        super().__init__()
        self.ce_loss_7 = nn.CrossEntropyLoss(weight=class_weights_7)
        self.ce_loss_4 = nn.CrossEntropyLoss(weight=class_weights_4)
        self.ce_loss_3 = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, outputs, targets):
        loss_7 = self.ce_loss_7(outputs['label_7_logits'], targets['label_7'])
        loss_4 = self.ce_loss_4(outputs['label_4_logits'], targets['label_4'])
        loss_3 = self.ce_loss_3(outputs['label_3_logits'], targets['label_3'])
        loss_A = self.mse_loss(outputs['arousal'], targets['arousal'])
        loss_V = self.mse_loss(outputs['valence'], targets['valence'])
        
        # 权重: 7类主导
        total_loss = 1.0 * loss_7 + 0.5 * loss_4 + 0.3 * loss_3 + 0.1 * loss_A + 0.1 * loss_V
        
        return {
            'total_loss': total_loss,
            'loss_7': loss_7.item(),
            'loss_4': loss_4.item(),
        }

print("数据集和损失函数定义完成!")

## 6. 加载数据

In [ ]:
from sklearn.model_selection import train_test_split

# 加载标签
print(f"加载标签: {LABELS_CSV}")
label_df = pd.read_csv(LABELS_CSV)
print(f"总样本数: {len(label_df)}")

# 显示7类分布
print("\n7类标签分布:")
emotion_names = {0: '愤怒', 1: '焦虑', 2: '快乐', 3: '悲伤', 4: '失望', 5: '支持', 6: '平静'}
for label_id in sorted(label_df['label_7_emotion'].unique()):
    count = len(label_df[label_df['label_7_emotion'] == label_id])
    print(f"  [{label_id}] {emotion_names[label_id]}: {count} ({count/len(label_df)*100:.1f}%)")

# 加载缓存
print(f"\n加载缓存: {CACHE_FILE}")
cached_features = torch.load(CACHE_FILE)
print(f"缓存样本数: {len(cached_features)}")

# 划分数据集
train_df, test_df = train_test_split(label_df, test_size=0.2, random_state=42)
print(f"\n训练集: {len(train_df)}, 测试集: {len(test_df)}")

In [ ]:
# 创建数据加载器
BATCH_SIZE = 32

train_dataset = CachedAudioDatasetV2(train_df, cached_features)
test_dataset = CachedAudioDatasetV2(test_df, cached_features)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                          shuffle=True, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"训练批次: {len(train_loader)}, 测试批次: {len(test_loader)}")

## 7. 创建模型和优化器

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建模型
model = AudioHMTLClassifier().to(device)

# 7类权重 (根据分布)
CLASS_WEIGHTS_7 = torch.tensor([
    4911 / (7 * 168),   # 愤怒
    4911 / (7 * 135),   # 焦虑
    4911 / (7 * 768),   # 快乐
    4911 / (7 * 49),    # 悲伤
    4911 / (7 * 58),    # 失望
    4911 / (7 * 671),   # 支持
    4911 / (7 * 3062),  # 平静
]).to(device)

CLASS_WEIGHTS_4 = torch.tensor([0.8466, 4.0917, 11.2874, 0.4023]).to(device)

# 损失函数
criterion = HMTLAudioLossV2(
    class_weights_7=CLASS_WEIGHTS_7,
    class_weights_4=CLASS_WEIGHTS_4
)

# 分层学习率优化器
LEARNING_RATE = 5e-5

backbone_params = list(model.wav2vec2.parameters())
head_params = (list(model.classifier_7.parameters()) +
               list(model.classifier_4.parameters()) +
               list(model.classifier_3.parameters()) +
               list(model.regressor_A.parameters()) +
               list(model.regressor_V.parameters()) +
               list(model.dim_reducer.parameters()))

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE},
    {'params': head_params, 'lr': LEARNING_RATE * 10}
])

print("模型和优化器创建完成!")
print(f"  Backbone LR: {LEARNING_RATE}")
print(f"  Head LR: {LEARNING_RATE * 10}")

## 8. 训练

In [ ]:
from tqdm import tqdm

def evaluate(model, dataloader, criterion, device):
    model.eval()
    correct_7 = 0
    correct_4 = 0
    total = 0
    
    with torch.no_grad():
        for batch in dataloader:
            if batch is None:
                continue
            
            features = batch['features'].to(device)
            targets = {k: v.to(device) for k, v in batch.items() if k != 'features'}
            
            outputs = model(features)
            
            pred_7 = outputs['label_7_logits'].argmax(dim=1)
            pred_4 = outputs['label_4_logits'].argmax(dim=1)
            
            correct_7 += (pred_7 == targets['label_7']).sum().item()
            correct_4 += (pred_4 == targets['label_4']).sum().item()
            total += targets['label_7'].size(0)
    
    return {
        'acc_7': correct_7 / total if total > 0 else 0,
        'acc_4': correct_4 / total if total > 0 else 0,
    }

print("评估函数定义完成!")

In [ ]:
NUM_EPOCHS = 15
best_acc_7 = 0

print("="*60)
print("开始训练 Audio HMTL V2")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        if batch is None:
            continue
        
        features = batch['features'].to(device)
        targets = {k: v.to(device) for k, v in batch.items() if k != 'features'}
        
        optimizer.zero_grad()
        outputs = model(features)
        loss_dict = criterion(outputs, targets)
        
        loss_dict['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss_dict['total_loss'].item()
        pbar.set_postfix({'loss': f"{loss_dict['total_loss'].item():.4f}"})
    
    # 评估
    eval_result = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1}: Loss={epoch_loss/len(train_loader):.4f}, "
          f"Acc_7={eval_result['acc_7']*100:.2f}%, "
          f"Acc_4={eval_result['acc_4']*100:.2f}%")
    
    # 保存最佳模型
    if eval_result['acc_7'] > best_acc_7:
        best_acc_7 = eval_result['acc_7']
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型: {best_acc_7*100:.2f}%")

print("\n" + "="*60)
print(f"训练完成! 最佳 7类准确率: {best_acc_7*100:.2f}%")
print(f"模型保存到: {MODEL_SAVE_PATH}")
print("="*60)

## 9. 测试模型输出格式

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

# 测试一个样本
test_batch = next(iter(test_loader))
features = test_batch['features'][:1].to(device)

with torch.no_grad():
    outputs = model(features)

print("模型输出格式:")
for key, value in outputs.items():
    print(f"  {key}: shape={value.shape}")

print("\n示例输出:")
print(f"  label_7_logits: {outputs['label_7_logits'][0].cpu().numpy()}")
print(f"  label_7 预测: {outputs['label_7_logits'].argmax(dim=1).item()} ({emotion_names[outputs['label_7_logits'].argmax(dim=1).item()]})")
print(f"  arousal: {outputs['arousal'][0].item():.4f}")
print(f"  valence: {outputs['valence'][0].item():.4f}")